In [ ]:
import jax

jax.config.update('jax_platform_name', 'cpu')
jax.config.update("jax_enable_x64", True)
import numpy as np
from learn_s_hat_toy import make_srs
from gould_2026.plotting import paper_plot_context, Palette
from gould_2026.datasets import ArrayWithTime
from gould_2026.datasets import NestDynamicsUFunction
import scipy.stats
from io import StringIO
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import seaborn as sns

rng = np.random.default_rng(0)

In [ ]:
output_1_learn_toy_manifold = None
output_2_stat_text = None
u_function = NestDynamicsUFunction.curvy_spins


In [ ]:
output_2_stat_text = Path(output_2_stat_text) if output_2_stat_text is not None else None

u_function = NestDynamicsUFunction(u_function)

In [ ]:
srs = make_srs(rng, n_runs=1, show_tqdm=True, add_s_hat_error_function=True, u_function=u_function)

del srs['ignoring stim samples']

In [ ]:
fig, ax = plt.subplots()


In [ ]:
dfs = []
funcs = set()
for k, sr_list in srs.items():
    for i, sr in enumerate(sr_list):
        sub_df = pd.DataFrame(sr.log['s_hat_error'])
        funcs = funcs.union(set(sub_df.columns))
        sub_df['sr_i'] = i
        sub_df['sr_type'] = k
        sub_df['stim_i'] = np.arange(len(sub_df))
        dfs.append(sub_df)

df = pd.concat(dfs, ignore_index=True)

funcs = list(funcs)
func = funcs[0]
df['stim_t'] = df[func].apply(lambda x: x.t)

In [ ]:
def f(e):
    return scipy.stats.bootstrap((e**2).mean(axis=1)[None,:], np.mean, confidence_level=0.95, n_resamples=10000, method='percentile')

for func in funcs:
    df[f'{func}_mse'] = df[func].apply(lambda x: np.mean(x**2))
    df[f'{func}_intervals'] = df[func].apply(f)


In [ ]:
sns.scatterplot(data=df, x='stim_i',y = 'stim_t', hue='sr_type', style='sr_i', alpha=.5)

In [ ]:
sns.scatterplot(data=df, x='stim_t',y = f'{func}_mse', hue='sr_type', markers='sr_i', alpha=.5)

In [ ]:
fig, axs = plt.subplots(ncols=len(funcs), figsize=(4*len(funcs), 4), sharey=True)

for ax, func in zip(axs.flatten(), funcs):
    for sr_type in df['sr_type'].unique():
        for sr_i in df['sr_i'].unique():
            sub_df = df[(df['sr_type'] == sr_type) & (df['sr_i'] == sr_i)]

            high = sub_df[f'{func}_intervals'].apply(lambda x: x.confidence_interval.high)
            med = sub_df[f'{func}_mse']
            low = sub_df[f'{func}_intervals'].apply(lambda x: x.confidence_interval.low)

            ax.fill_between(sub_df['stim_t'], low, high, alpha=.2)
            ax.plot(sub_df['stim_t'], med, alpha=1)
    ax.set_title(func)

In [ ]:
fig, axs = plt.subplots(ncols=3, figsize=(3*4, 4), sharey=True)

for ax, func in zip(axs.flatten(), ['curvy', 'curvy_flipped', 'curvy_flips', ]):
    for sr_type in df['sr_type'].unique():
        for sr_i in df['sr_i'].unique():
            c = Palette.blind if sr_type == 'unaware of stim' else Palette.stim_regressed

            sub_df = df[(df['sr_type'] == sr_type) & (df['sr_i'] == sr_i)]

            high = sub_df[f'{func}_intervals'].apply(lambda x: x.confidence_interval.high)
            med = sub_df[f'{func}_mse']
            low = sub_df[f'{func}_intervals'].apply(lambda x: x.confidence_interval.low)

            ax.fill_between(sub_df['stim_t'], low, high, alpha=.2, color=c)
            ax.plot(sub_df['stim_t'], med, alpha=1, color=c)
    ax.set_title(func)


In [ ]:
fig, ax = plt.subplots()

func = 'curvy'
for sr_type in df['sr_type'].unique():
    for sr_i in df['sr_i'].unique():
        c = Palette.blind if sr_type == 'unaware of stim' else Palette.stim_regressed

        sub_df = df[(df['sr_type'] == sr_type) & (df['sr_i'] == sr_i)]

        high = sub_df[f'{func}_intervals'].apply(lambda x: x.confidence_interval.high)
        med = sub_df[f'{func}_mse']
        low = sub_df[f'{func}_intervals'].apply(lambda x: x.confidence_interval.low)

        ax.fill_between(sub_df['stim_t'], low, high, alpha=.2, color=c)
        ax.plot(sub_df['stim_t'], med, alpha=1, color=c)

if output_1_learn_toy_manifold is not None:
    fig.savefig(output_1_learn_toy_manifold)


In [ ]:
shes = np.array(shes)
test_sample = 9
aware, unaware = shes[:, test_sample,]
test_result = scipy.stats.wilcoxon(unaware, aware)
test_string = StringIO()
test_string.write(f"'learning from stim' vs 'unaware of stim' at {test_sample = }\n")
test_string.write(f"{test_result = }\n")
test_string.write(f"Δ = {np.mean(unaware) - np.mean(aware):.3f} {np.mean(unaware)=:.3f} {np.mean(aware)=:.3f}\n")

if output_2_stat_text is not None:
    with output_2_stat_text.open('w') as fhan:
        fhan.write(test_string.getvalue())
